### Ingesting Races File

1) I'll reading file using spark
2) Adding metadata columns or audit columns such as:

    -- Source File

    --Ingestion Timestamp

3) Once standandardized, I will be writting them to Bronze Table    

Reading the CSV file

In [0]:
%run "../00-Common/01. Env Config"

In [0]:
%run "../00-Common/02. Bronze-helper_func"

In [0]:
Source_file=f"{landing_folder_path}/races.csv"
Table_name=f"{catalog_name}.{bronze_schema}.races"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType


race_Schema=StructType([
    StructField('season', IntegerType()),
    StructField('round', IntegerType()),
    StructField('url', StringType()),
    StructField('racesName', StringType()),
    StructField('date', DateType()),
    StructField('circuitId', StringType())
])

race_Schema


In [0]:
races_df=(
                spark.read.format('csv')
                .option('header',True)
                #.option('inferSchema','true')
                .option('mode','FAILFAST')      # PERMISSIVE, DROPMALFORMED, FAILFAST
                .schema(race_Schema)
                .load(Source_file))

In [0]:
display(races_df)

2) Adding Metadata

In [0]:
races_final_df= add_ingestion_metadata(races_df)

In [0]:
display(races_final_df)

3) Witting data to Bronze layer/delta table

In [0]:
(
    races_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(Table_name)
)

In [0]:
display(spark.table(Table_name))